# RQ5, Part 1: Cross-Domain Point-Estimate Synthesis (Four-Project Version)

**This notebook replaces the previous version of `RQ5_1_Point_Estimate_Synthesis.ipynb`,** which averaged RQ1's effect size across three groups (Camel/Hadoop combined, Kafka, Tika). That grouping predates the RQ1 correction that established Camel, Hadoop, Kafka, and Tika as four independently tested projects (see `RQ1_6_Interpretation_and_Robustness_UPDATED.ipynb`). This version uses the same four-project definition throughout, so its output matches the final report exactly.

Requires: `camel_real_mined_dataset.csv`, `hadoop_real_mined_dataset.csv`, `kafka_real_mined_dataset.csv`, `tika_real_mined_dataset.csv`, `rq4_real_sec_edgar_dataset_FINAL.csv` (all in `data/cleaned/`).

In [1]:
# --- Setup: make this notebook runnable standalone in Colab or locally ---
import os

REPO_URL = "https://github.com/daljeetkaurJohar/qm640-governance-analytics.git"
REPO_DIR = "qm640-governance-analytics"

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    # Colab starts in /content with no repo checked out -- clone it once,
    # then cd into it so the relative "data/cleaned/..." paths below resolve.
    if not os.path.isdir(f"/content/{REPO_DIR}"):
        get_ipython().system(f"git clone --depth 1 {REPO_URL} /content/{REPO_DIR}")
    os.chdir(f"/content/{REPO_DIR}")
else:
    # Running locally: assume this notebook is being run from notebooks/ inside
    # the cloned repo (its normal location) and step up to the repo root, where
    # the "data/cleaned/..." paths below expect to be run from.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print("Working directory:", os.getcwd())
assert os.path.isfile("data/cleaned/camel_real_mined_dataset.csv"), (
    "camel_real_mined_dataset.csv not found -- check that the repo cloned/changed "
    "directory correctly above, or that you are running from the repo root."
)


Cloning into '/content/qm640-governance-analytics'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 107 (delta 22), reused 59 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 7.20 MiB | 8.16 MiB/s, done.
Resolving deltas: 100% (22/22), done.
Working directory: /content/qm640-governance-analytics


In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

def cohens_f_squared(r2_full, r2_reduced):
    return (r2_full - r2_reduced) / (1 - r2_full)

def interpret(f2):
    if f2 < 0.02:
        return "negligible"
    elif f2 < 0.15:
        return "small"
    elif f2 < 0.35:
        return "medium"
    return "large"

def rq1_project_f2(path, target):
    df = pd.read_csv(path)
    df["era_binary"] = (df["era"] == "ai_era").astype(int)
    full = smf.logit(
        f"{target} ~ (loc + cyclomatic_complexity + num_functions + "
        f"num_files_changed) * era_binary", data=df).fit(disp=0)
    reduced = smf.logit(
        f"{target} ~ loc + cyclomatic_complexity + num_functions + "
        f"num_files_changed + era_binary", data=df).fit(disp=0)
    r2f = 1 - (full.llf / full.llnull)
    r2r = 1 - (reduced.llf / reduced.llnull)
    return len(df), cohens_f_squared(r2f, r2r)


## QA domain: four independent RQ1 projects

In [3]:
projects = [
    ("Camel", "data/cleaned/camel_real_mined_dataset.csv", "defect_prone_strict"),
    ("Hadoop", "data/cleaned/hadoop_real_mined_dataset.csv", "defect_prone_strict"),
    ("Kafka", "data/cleaned/kafka_real_mined_dataset.csv", "defect_prone"),
    ("Tika", "data/cleaned/tika_real_mined_dataset.csv", "defect_prone"),
]

print("=== RQ1 effect sizes (4 independent projects) ===")
rq1_f2 = {}
for name, path, target in projects:
    n, f2 = rq1_project_f2(path, target)
    rq1_f2[name] = f2
    print(f"  {name}: N={n}, f2={f2:.4f}")

qa_domain_f2 = float(np.mean(list(rq1_f2.values())))
print(f"\nQA domain average (4 projects): {qa_domain_f2:.4f} ({interpret(qa_domain_f2)})")


=== RQ1 effect sizes (4 independent projects) ===
  Camel: N=12598, f2=0.0036
  Hadoop: N=6038, f2=0.0041
  Kafka: N=433, f2=0.0178
  Tika: N=395, f2=0.0038

QA domain average (4 projects): 0.0073 (negligible)


## Audit domain: RQ4 effect size (full sample, N=113)

In [4]:
def consolidate_weakness(cat):
    cat = str(cat)
    if "Revenue" in cat:
        return "Revenue Recognition"
    if "ITGC" in cat:
        return "ITGC"
    if "Complex" in cat or "Warrant" in cat or "Instrument" in cat:
        return "Complex Transactions/Instruments"
    if "Control Environment" in cat or "Staffing" in cat or "Risk Assessment" in cat or "Segregation" in cat:
        return "Control Environment/Staffing"
    return "Other"

def consolidate_industry(ind):
    ind = str(ind)
    if "Technology" in ind:
        return "Technology"
    if "Biotech" in ind or "Healthcare" in ind:
        return "Biotech/Healthcare"
    if "Manufacturing" in ind or "Industrial" in ind or "Aerospace" in ind or "Mining" in ind:
        return "Manufacturing/Industrial"
    if "SPAC" in ind:
        return "SPAC"
    if "Energy" in ind:
        return "Energy"
    if "Financial" in ind or "Insurance" in ind or "Real Estate" in ind:
        return "Financial/Real Estate"
    return "Media/Consumer/Other"

rq4 = pd.read_csv("data/cleaned/rq4_real_sec_edgar_dataset_FINAL.csv", parse_dates=["disclosure_date", "remediation_date"])
rq4["weakness_group"] = rq4["weakness_category"].apply(consolidate_weakness)
rq4["industry_group"] = rq4["industry"].apply(consolidate_industry)
rq4["disclosure_year"] = rq4["disclosure_date"].dt.year

full_model = smf.ols("remediation_days ~ C(weakness_group) + C(industry_group) + disclosure_year", data=rq4).fit()
reduced_model = smf.ols("remediation_days ~ C(weakness_group)", data=rq4).fit()
audit_domain_f2 = cohens_f_squared(full_model.rsquared, reduced_model.rsquared)

print(f"RQ4 (all N=113): R2_full={full_model.rsquared:.4f}, R2_reduced={reduced_model.rsquared:.4f}")
print(f"Audit domain effect size: {audit_domain_f2:.4f} ({interpret(audit_domain_f2)})")


RQ4 (all N=113): R2_full=0.0791, R2_reduced=0.0216
Audit domain effect size: 0.0624 (small)


## Cross-domain comparison

In [5]:
gap = abs(qa_domain_f2 - audit_domain_f2)
threshold = 0.10

print("=== CROSS-DOMAIN COMPARISON (4-project QA definition) ===")
print(f"QA domain (avg of 4 independent RQ1 projects): {qa_domain_f2:.4f}")
print(f"Audit domain (RQ4, full N=113): {audit_domain_f2:.4f}")
print(f"Absolute gap: {gap:.4f}")
print(f"Pre-specified equivalence threshold: {threshold}")
print(f"Point estimate {'below' if gap < threshold else 'above'} threshold")
print()
print("NOTE: this point estimate alone is NOT the final, honest answer -- see")
print("RQ5_2_Bootstrap_CI.ipynb for the real uncertainty around this gap, and note")
print("that RQ1 uses a logistic-regression pseudo-R2 while RQ4 uses ordinary OLS")
print("R2, so this comparison is exploratory rather than directly equivalent.")


=== CROSS-DOMAIN COMPARISON (4-project QA definition) ===
QA domain (avg of 4 independent RQ1 projects): 0.0073
Audit domain (RQ4, full N=113): 0.0624
Absolute gap: 0.0551
Pre-specified equivalence threshold: 0.1
Point estimate below threshold

NOTE: this point estimate alone is NOT the final, honest answer -- see
RQ5_2_Bootstrap_CI.ipynb for the real uncertainty around this gap, and note
that RQ1 uses a logistic-regression pseudo-R2 while RQ4 uses ordinary OLS
R2, so this comparison is exploratory rather than directly equivalent.
